# Setup

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path


### change this value to false if you want to display plots for files already in data/bike and data/drone. Change it to true if you want to create new csv files ready for plotting from the files inside logs/bike logs/drone

In [ ]:
create_new_csv = False

### Code used to save plots into plots folder

In [ ]:
def save_plot(filename, info):
    path = f"plots/{info['Stage']}_{filename}.png"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.savefig(path, dpi=150, bbox_inches="tight")

In [ ]:
# Function to load in the files
def load_data(file_name):
    bike_frames = []
    drone_frames = []

    n_bikes = len(os.listdir(f"../logs/bike/{file_name}"))
    n_drones = len(os.listdir(f"../logs/drone/{file_name}"))

    name_dict = {}
    name = ""

    for i in range(1, n_bikes + 1):
        bike_file = f"../logs/bike/{file_name}/bike_{i}.csv"
        with open(bike_file, "r") as f:
            name = f.readline().strip()
            
        if os.path.exists(bike_file):
            bike_df = pd.read_csv(bike_file, header=1, skipinitialspace=True)
            bike_df.insert(loc=0, column="ID", value=i)
            bike_frames.append(bike_df)

    for i in range(1, n_drones + 1):
        drone_file = f"../logs/drone/{file_name}/drone_{i}.csv"
        if os.path.exists(drone_file):
            drone_df = pd.read_csv(drone_file, header=1, skipinitialspace=True)
            drone_df.insert(loc=0, column="ID", value=i)
            drone_frames.append(drone_df)

    bike_data = pd.concat(bike_frames, ignore_index=True) if bike_frames else pd.DataFrame()
    drone_data = pd.concat(drone_frames, ignore_index=True) if drone_frames else pd.DataFrame()

    name = name.split()
    for i, key in enumerate(name):
        if i % 2 == 0:
            name_dict[key] = name[i+1].replace(",", "").replace(".json", "")

    bike_data.attrs["run_info"] = name_dict
    drone_data.attrs["run_info"] = name_dict
    
    return bike_data, drone_data

In [ ]:
def parse_ids(value):
    return set(map(int, str(value).replace("[", "").replace("]", "").split()))

def add_nearby_info(bike_data, drone_data, radius):
    bike_data = bike_data.copy()
    drone_data = drone_data.copy()

    bike_data["nearby_drones"] = 0
    bike_data["min_distance"] = 0.0
    bike_data["in_camera_shot"] = False

    drone_data["nearby_bikes"] = 0
    drone_data["min_distance"] = 0.0
    drone_data["bikes_in_camera"] = 0

    bike_nearby_drones = np.zeros(len(bike_data), dtype=int)
    bike_min_distance = np.full(len(bike_data), np.inf)
    bike_in_camera = np.zeros(len(bike_data), dtype=bool)

    drone_nearby_bikes = np.zeros(len(drone_data), dtype=int)
    drone_min_distance = np.full(len(drone_data), np.inf)
    drone_bikes_in_camera = np.zeros(len(drone_data), dtype=int)

    bike_groups = bike_data.groupby("Timestep", sort=False).indices
    drone_groups = drone_data.groupby("Timestep", sort=False).indices

    bike_ids = bike_data["ID"].to_numpy()
    drone_bike_ids_raw = drone_data["Bikes-ID"].to_numpy()

    bike_pos = bike_data[["Pos x", "Pos y", "Pos z"]].to_numpy(dtype=float)
    drone_pos = drone_data[["Pos x", "Pos y", "Pos z"]].to_numpy(dtype=float)

    common_timesteps = set(bike_groups.keys()) & set(drone_groups.keys())

    for timestep in common_timesteps:
        b_idx = bike_groups[timestep]
        d_idx = drone_groups[timestep]

        b_positions = bike_pos[b_idx]
        d_positions = drone_pos[d_idx]

        diff = b_positions[:, None, :] - d_positions[None, :, :]

        nearby_cube = np.all(np.abs(diff) <= radius, axis=2)

        distances = np.sqrt(np.sum(diff ** 2, axis=2))

        bike_nearby_drones[b_idx] = nearby_cube.sum(axis=1)
        drone_nearby_bikes[d_idx] = nearby_cube.sum(axis=0)

        bike_min_distance[b_idx] = distances.min(axis=1)
        drone_min_distance[d_idx] = distances.min(axis=0)

        current_bike_ids = bike_ids[b_idx]

        for local_d_pos, global_d_idx in enumerate(d_idx):
            visible_ids = parse_ids(drone_bike_ids_raw[global_d_idx])

            visible_mask = np.isin(current_bike_ids, list(visible_ids))

            drone_bikes_in_camera[global_d_idx] = visible_mask.sum()

            bike_in_camera[b_idx[visible_mask]] = True

    bike_data["nearby_drones"] = bike_nearby_drones
    bike_data["min_distance"] = bike_min_distance
    bike_data["in_camera_shot"] = bike_in_camera

    drone_data["nearby_bikes"] = drone_nearby_bikes
    drone_data["min_distance"] = drone_min_distance
    drone_data["bikes_in_camera"] = drone_bikes_in_camera

    return bike_data, drone_data

In [ ]:
def setup():
    dirs = os.listdir("../logs/bike")
    # remove non-run directories if they exist
    dirs = [d for d in dirs]
    run = len(dirs)

    all_bike_data = []
    all_drone_data = []

    for i in range(0, run):
        bike_data, drone_data = load_data(dirs[i])
        run_info = bike_data.attrs["run_info"]

        bike_data, drone_data = add_nearby_info(
            bike_data,
            drone_data,
            int(run_info["Size:"])
        )

        bike_data.attrs["run_info"] = run_info
        drone_data.attrs["run_info"] = run_info

        all_bike_data.append(bike_data)
        all_drone_data.append(drone_data)

    return all_bike_data, all_drone_data

## Creates new csv files used for plotting, for each file in the folders logs/bikes and logs/drones

In [ ]:
if create_new_csv:
    bike_data_list, drone_data_list = [], []
    bike_data_list, drone_data_list = setup()
    runs = len(bike_data_list)
    print(f"Loaded {runs} runs of data.")

    def save_run_csv(df, path, info):
        if not os.path.exists(os.path.dirname(path)):
            os.makedirs(os.path.dirname(path))
            
        with open(path, "w") as f:
            f.write(f"# Bikes:{info['Bikes:']}\n")
            f.write(f"# Drones:{info['Drones:']}\n")
            f.write(f"# Stage:{info['Stage:']}\n")
            f.write("# DATA\n")
            df.to_csv(f, index=False)

    for i in range(1, runs + 1):
        bike_df = bike_data_list[i - 1]
        drone_df = drone_data_list[i - 1]

        info = bike_df.attrs["run_info"]

        # Save both
        save_run_csv(
            bike_df,
            f"data/bike/{info['Stage:']}_bikes_{info['Bikes:']}_drones_{info['Drones:']}_run_{i}.csv",
            info
        )

        save_run_csv(
            drone_df,
            f"data/drone/{info['Stage:']}_bikes_{info['Bikes:']}_drones_{info['Drones:']}_run_{i}.csv",
            info
        )


# Plot data

In [ ]:
bike_folder = Path("data/bike")
num_files = len(list(bike_folder.glob("*.csv")))

In [ ]:
def load_run_csv(path):
    metadata = {}
    data_start = 0

    with open(path, "r") as f:
        for idx, line in enumerate(f):
            if line.startswith("# DATA"):
                data_start = idx + 1
                break

            if line.startswith("#"):
                key, value = line[1:].strip().split(":", 1)
                metadata[key.strip()] = value.strip()

    df = pd.read_csv(path, skiprows=data_start)
    return df, metadata


def get_run_files(folder):
    return sorted(Path(folder).glob("*.csv"))


def plot_title(base_title, info):
    return (
        f"{base_title}\n"
        f"Bikes: {info['Bikes']} - "
        f"Drones: {info['Drones']} - "
        f"Stage: {info['Stage']}"
    )

#### Average distance from drone to nearest bike

In [ ]:
for i, path in enumerate(get_run_files("data/drone")):
    df, info = load_run_csv(path)
    avg_dist = df.groupby("Timestep")["min_distance"].agg(["mean", "min", "max"]).reset_index()

    sns.lineplot(data=avg_dist, x="Timestep", y="mean")

    plt.title(plot_title("Average distance from drone to nearest bike over time", info))
    plt.xlabel("Timestep")
    plt.ylabel("Average Min Distance")
    save_plot(f"run_{i+1}_avg_dist_to_nearest_bike", info)
    plt.show()

#### Average distance from bike to nearest drone

In [ ]:
for i, path in enumerate(get_run_files("data/bike")):
    df, info = load_run_csv(path)

    avg_dist = df.groupby("Timestep")["min_distance"].agg(["mean", "min", "max"]).reset_index()

    sns.lineplot(data=avg_dist, x="Timestep", y="mean")

    plt.title(plot_title("Average distance from bike to nearest drone over time", info))
    plt.xlabel("Timestep")
    plt.ylabel("Average Min Distance")
    save_plot(f"run_{i+1}_avg_dist_to_nearest_drone", info)
    plt.show()

### Percentage of bikes in range of a drone


In [ ]:
for i, path in enumerate(get_run_files("data/bike")):
    df, info = load_run_csv(path)

    # Number of active bikes per timestep
    active_bikes = (
        df.groupby("Timestep")["ID"]
        .nunique()
        .reset_index(name="active_bikes")
    )

    # Percentage of bikes in drone range
    percentage_per_time = (
        df.groupby("Timestep")["nearby_drones"]
        .apply(lambda x: (x > 0).mean() * 100)
        .reset_index(name="bikes_in_range_of_drones")
    )

    # Merge both datasets
    plot_df = percentage_per_time.merge(active_bikes, on="Timestep")

    # Create figure
    fig, ax1 = plt.subplots()

    # Percentage line
    sns.lineplot(
        data=plot_df,
        x="Timestep",
        y="bikes_in_range_of_drones",
        ax=ax1,
        label="% of bikes"
    )

    ax1.set_ylabel("Bikes in range (%)")
    ax1.set_ylim(0, 101)
    ax1.yaxis.set_major_formatter(lambda x, _: f"{x:.0f}%")

    # Second axis for active bikes
    ax2 = ax1.twinx()

    sns.lineplot(
        data=plot_df,
        x="Timestep",
        y="active_bikes",
        ax=ax2,
        color="orange",
        label="Active bikes"
    )

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")
    ax2.get_legend().remove()

    # Title
    plt.title(plot_title("Percentage of bikes in range of a drone", info))

    save_plot(f"run_{i+1}_percent_of_bikes_in_range_of_drone", info)
    plt.show()

### Percentage of bikes in camera shot of a drone

In [ ]:
SMOOTH_WINDOW = 500

for i, path in enumerate(get_run_files("data/bike")):
    df, info = load_run_csv(path)
    df["in_camera_shot"] = (
        df["in_camera_shot"]
        .astype(str)
        .str.lower()
        == "true"
    )

    active_bikes = (
        df.groupby("Timestep")
        .size()
        .reset_index(name="active_bikes")
    )
    percentage_per_time = (
        df.groupby("Timestep")["in_camera_shot"]
        .mean()
        .mul(100)
        .reset_index(name="bikes_in_camera_shot")
    )
    plot_df = percentage_per_time.merge(active_bikes, on="Timestep")
    plot_df["bikes_in_camera_shot_smooth"] = plot_df["bikes_in_camera_shot"].rolling(window=SMOOTH_WINDOW, center=True).mean()
    plot_df["active_bikes_smooth"] = plot_df["active_bikes"].rolling(window=SMOOTH_WINDOW, center=True).mean()

    fig, ax1 = plt.subplots()

    ax1.plot(plot_df["Timestep"], plot_df["bikes_in_camera_shot"], alpha=0.15, color="steelblue")
    ax1.plot(plot_df["Timestep"], plot_df["bikes_in_camera_shot_smooth"], color="steelblue", label="% in camera shot")
    ax1.set_ylabel("Bikes in camera shot (%)")
    ax1.set_ylim(0, 101)
    ax1.yaxis.set_major_formatter(lambda x, _: f"{x:.0f}%")

    ax2 = ax1.twinx()
    ax2.plot(plot_df["Timestep"], plot_df["active_bikes"], alpha=0.15, color="orange")
    ax2.plot(plot_df["Timestep"], plot_df["active_bikes_smooth"], color="orange", label="Active bikes")
    ax2.set_ylabel("Active bikes")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")

    plt.title(plot_title("Percentage of bikes inside camera shot of a drone", info))
    plt.tight_layout()
    save_plot(f"run_{i+1}_percent_of_bikes_in_camera_of_drone", info)
    plt.show()

### Amount of bikes not within a camera each timestep

In [ ]:
SMOOTH_WINDOW = 500

for i, path in enumerate(get_run_files("data/bike")):
    df, info = load_run_csv(path)
    df["in_camera_shot"] = df["in_camera_shot"].astype(str).str.lower() == "true"
    bikes_outside = (
        df.groupby("Timestep")["in_camera_shot"]
        .apply(lambda x: (~x).sum())
        .reset_index(name="bikes_outside_camera")
    )
    bikes_outside["smooth"] = bikes_outside["bikes_outside_camera"].rolling(window=SMOOTH_WINDOW, center=True).mean()

    fig, ax = plt.subplots()
    ax.plot(bikes_outside["Timestep"], bikes_outside["bikes_outside_camera"], alpha=0.15, color="steelblue")
    ax.plot(bikes_outside["Timestep"], bikes_outside["smooth"], color="steelblue")
    ax.set_ylabel("Bikes outside camera")
    ax.set_xlabel("Timestep")
    plt.title(plot_title("Number of bikes outside camera shot", info))
    plt.tight_layout()
    save_plot(f"run_{i+1}_amount_of_bikes_in_camera_of_drone", info)
    plt.show()

### Collisions over time

In [ ]:
for i, path in enumerate(get_run_files("data/drone")):
    df, info = load_run_csv(path)
    collisions_at_time = df.groupby("Timestep")["Collisions"].sum().reset_index()

    collisions_at_time["Collisions"] = collisions_at_time["Collisions"] / 2
    collisions_at_time["Cumulative_collisions"] = collisions_at_time["Collisions"].cumsum()

    sns.lineplot(
        data=collisions_at_time, x="Timestep", y="Cumulative_collisions"
    )

    plt.title(plot_title("Cumulative Collisions per Timestep Over Time", info))
    save_plot(f"run_{i+1}_cumulative_collisions_drones", info)
    plt.show()


### Average amount of bikes in range of a drone over time

In [ ]:
for i, path in enumerate(get_run_files("data/drone")):
    df, info = load_run_csv(path)
    nearby_bikes_stats = df.groupby("Timestep")["nearby_bikes"].agg(["mean", "min", "max"]).reset_index()

    sns.lineplot(data=nearby_bikes_stats, x="Timestep", y="mean")

    plt.title(plot_title("Average number of bikes in range per drone over time", info))
    plt.ylabel("Average amount nearby bikes")
    save_plot(f"run_{i+1}_average_number_of_bikes_in_range_pr_drone", info)
    plt.show()

### Average amount of bikes inside camera shot of a drone

In [ ]:
SMOOTH_WINDOW = 500

for i, path in enumerate(get_run_files("data/drone")):
    df, info = load_run_csv(path)
    camera_stats = (
        df.groupby("Timestep")["bikes_in_camera"]
        .agg(["mean", "min", "max"])
        .reset_index()
    )
    camera_stats["mean_smooth"] = camera_stats["mean"].rolling(window=SMOOTH_WINDOW, center=True).mean()

    fig, ax = plt.subplots()
    ax.plot(camera_stats["Timestep"], camera_stats["mean"], alpha=0.15, color="steelblue")
    ax.plot(camera_stats["Timestep"], camera_stats["mean_smooth"], color="steelblue", label="Mean (smoothed)")
    ax.set_ylabel("Bikes in camera per drone")
    ax.set_xlabel("Timestep")
    ax.legend(loc="upper right")
    plt.title(plot_title("Average number of bikes inside camera shot per drone", info))
    plt.tight_layout()
    save_plot(f"run_{i+1}_average_number_of_bikes_in_camera_pr_drone", info)
    plt.show()

### Amount of drones with no bikes in camera shot

In [ ]:
SMOOTH_WINDOW = 500

for i, path in enumerate(get_run_files("data/drone")):
    df, info = load_run_csv(path)
    no_camera_stats = (
        df.groupby("Timestep")["bikes_in_camera"]
        .apply(lambda x: (x == 0).sum())
        .reset_index(name="drones_with_no_bikes_in_camera")
    )
    no_camera_stats["smooth"] = no_camera_stats["drones_with_no_bikes_in_camera"].rolling(window=SMOOTH_WINDOW, center=True).mean()

    fig, ax = plt.subplots()
    ax.plot(no_camera_stats["Timestep"], no_camera_stats["drones_with_no_bikes_in_camera"], alpha=0.15, color="steelblue")
    ax.plot(no_camera_stats["Timestep"], no_camera_stats["smooth"], color="steelblue")
    ax.set_ylabel("Drones with empty camera")
    ax.set_xlabel("Timestep")
    plt.title(plot_title("Number of drones with no bikes in camera over time", info))
    plt.tight_layout()
    save_plot(f"run_{i+1}_number_of_drones_with_no_bikes_in_camera", info)
    plt.show()

### Detect lost drones

In [ ]:
LOST_FOR_N_STEPS = 10

for i, path in enumerate(get_run_files("data/drone")):
    df, info = load_run_csv(path)

    df["is_lost"] = (df["nearby_bikes"] == 0) & (df["bikes_in_camera"] == 0)

    confirmed_lost = pd.Series(False, index=df.index)
    for drone_id, drone_df in df.groupby("ID"):
        drone_df = drone_df.sort_values("Timestep")
        is_lost = drone_df["is_lost"]
        consecutive = (
            is_lost
            .astype(int)
            .groupby((is_lost != is_lost.shift()).cumsum())
            .transform("sum")
        )
        confirmed_lost.loc[drone_df.index] = is_lost & (consecutive >= LOST_FOR_N_STEPS)

    df["confirmed_lost"] = confirmed_lost

    lost_per_timestep = (
        df.groupby("Timestep")["confirmed_lost"]
        .sum()
        .reset_index(name="lost_drones")
    )

    n_drones = int(info["Drones"])

    fig, ax = plt.subplots()
    ax.fill_between(
        lost_per_timestep["Timestep"],
        lost_per_timestep["lost_drones"],
        alpha=0.4,
        color="red"
    )
    sns.lineplot(
        data=lost_per_timestep,
        x="Timestep",
        y="lost_drones",
        ax=ax,
        color="red"
    )
    ax.set_ylim(0, n_drones)
    ax.set_ylabel("Number of lost drones")
    ax.set_xlabel("Timestep")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.title(plot_title("Number of lost drones per timestep", info))
    plt.tight_layout()
    save_plot(f"run_{i+1}_number_of_lost_drones", info)
    plt.show()